In [1]:
# Cell 1 — Imports & config

In [1]:
import os
import json
import math
import random
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer

torch.manual_seed(42)
random.seed(42)

BASE_DIR = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational"

CONFIG = {
    "vocab_size": 50257,
    "max_seq_len": 512,
    "embed_dim": 512,
    "num_heads": 8,
    "num_layers": 12,
    "d_ff": 2730,
    "dropout": 0.1,

    "lora_r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.05,
    "target_modules": ["qkv_proj", "out_proj"],

    "batch_size": 8,
    "grad_accum_steps": 4,
    "lr": 2e-4,
    "warmup_steps": 200,
    "epochs": 5,
    "grad_clip": 0.8,
    "eval_every_steps": 200,
    "save_every_steps": 200,
    "patience": 3,  # tighter than Stage A's implicit patience — DailyDialog is more repetitive, overfits faster

    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

print("Device:", CONFIG["device"])
print(CONFIG)

Device: cuda
{'vocab_size': 50257, 'max_seq_len': 512, 'embed_dim': 512, 'num_heads': 8, 'num_layers': 12, 'd_ff': 2730, 'dropout': 0.1, 'lora_r': 32, 'lora_alpha': 64, 'lora_dropout': 0.05, 'target_modules': ['qkv_proj', 'out_proj'], 'batch_size': 8, 'grad_accum_steps': 4, 'lr': 0.0002, 'warmup_steps': 200, 'epochs': 5, 'grad_clip': 0.8, 'eval_every_steps': 200, 'save_every_steps': 200, 'patience': 3, 'device': 'cuda'}


In [2]:
# Cell 2 — Model architecture (same as Stage A — paste as-is)

In [3]:
class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

    def forward(self, x, seq_len):
        return self.cos_cached[:, :, :seq_len, :], self.sin_cached[:, :, :seq_len, :]

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(q, k, cos, sin):
    q = (q * cos) + (rotate_half(q) * sin)
    k = (k * cos) + (rotate_half(k) * sin)
    return q, k

class SwiGLU(nn.Module):
    def __init__(self, dim, d_ff, dropout):
        super().__init__()
        self.w1 = nn.Linear(dim, d_ff, bias=False)
        self.w2 = nn.Linear(dim, d_ff, bias=False)
        self.w3 = nn.Linear(d_ff, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w3(F.silu(self.w1(x)) * self.w2(x)))

class CausalSelfAttention(nn.Module):
    def __init__(self, dim, num_heads, max_seq_len, dropout):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv_proj = nn.Linear(dim, dim * 3, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.rope = RotaryEmbedding(self.head_dim, max_seq_len)
        self.dropout = dropout

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv_proj(x).reshape(B, T, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rope(x, T)
        q, k = apply_rope(q, k, cos, sin)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.out_proj(out)

class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = CausalSelfAttention(dim, num_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(dim)
        self.ffn = SwiGLU(dim, d_ff, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class ENGLlmModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.token_embed = nn.Embedding(cfg["vocab_size"], cfg["embed_dim"])
        self.blocks = nn.ModuleList([
            TransformerBlock(cfg["embed_dim"], cfg["num_heads"], cfg["d_ff"], cfg["max_seq_len"], cfg["dropout"])
            for _ in range(cfg["num_layers"])
        ])
        self.ln_f = nn.LayerNorm(cfg["embed_dim"])
        self.lm_head = nn.Linear(cfg["embed_dim"], cfg["vocab_size"], bias=False)

    def forward(self, input_ids):
        x = self.token_embed(input_ids)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.lm_head(x)

print("Model architecture defined.")

Model architecture defined.


In [4]:
# Cell 3 — LoRA wrapper (same as Stage A)

In [5]:
class LoRALinear(nn.Module):
    def __init__(self, base_linear, r, alpha, dropout):
        super().__init__()
        self.base = base_linear
        self.base.weight.requires_grad = False

        in_dim = base_linear.in_features
        out_dim = base_linear.out_features

        self.lora_A = nn.Parameter(torch.zeros(r, in_dim))
        self.lora_B = nn.Parameter(torch.zeros(out_dim, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

        self.scaling = alpha / r
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        base_out = self.base(x)
        lora_out = self.dropout(x) @ self.lora_A.T @ self.lora_B.T
        return base_out + lora_out * self.scaling

def apply_lora(model, target_modules, r, alpha, dropout):
    for block in model.blocks:
        if "qkv_proj" in target_modules:
            block.attn.qkv_proj = LoRALinear(block.attn.qkv_proj, r, alpha, dropout)
        if "out_proj" in target_modules:
            block.attn.out_proj = LoRALinear(block.attn.out_proj, r, alpha, dropout)
    return model

print("LoRA wrapper defined.")

LoRA wrapper defined.


In [6]:
# Cell 4 — Load the MERGED checkpoint as base, apply fresh LoRA

In [7]:
class GPTConfig:
    pass

MERGED_CKPT_PATH = os.path.join(BASE_DIR, "checkpoints", "stage_a_merged", "merged.pt")

model = ENGLlmModel(CONFIG)

checkpoint = torch.load(MERGED_CKPT_PATH, map_location="cpu", weights_only=False)
state_dict = checkpoint["model_state_dict"]

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

for p in model.parameters():
    p.requires_grad = False

model = apply_lora(model, CONFIG["target_modules"], CONFIG["lora_r"], CONFIG["lora_alpha"], CONFIG["lora_dropout"])
model.to(CONFIG["device"])

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} ({100*trainable/total:.2f}% of {total:,})")

Missing keys: []
Unexpected keys: []
Trainable params: 1,179,648 (1.02% of 115,570,688)


In [8]:
# Check this print carefully: missing/unexpected should be empty here too (or only RoPE buffers), same as your merge sanity check. If it's not clean here, something changed between the merge and this load — stop and check before continuing.

In [9]:
# Cell 5 — Dataset (train/val split from stage_b_train.jsonl)

In [10]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

STAGE_B_PATH = os.path.join(BASE_DIR, "data", "processed", "stage_b_train.jsonl")

class JsonlTextDataset(Dataset):
    def __init__(self, examples_text, tokenizer, max_len):
        self.examples = []
        for text in examples_text:
            full = text + tokenizer.eos_token
            ids = tokenizer.encode(full, truncation=True, max_length=max_len)
            self.examples.append(ids)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

def collate_fn(batch, pad_id, max_len):
    max_batch_len = min(max(len(x) for x in batch), max_len)
    input_ids, labels = [], []
    for ids in batch:
        ids = ids[:max_batch_len]
        pad_len = max_batch_len - len(ids)
        input_ids.append(ids + [pad_id] * pad_len)

        # Shift labels: label at position t is the token at position t+1
        shifted = ids[1:] + [-100] * (pad_len + 1)
        labels.append(shifted[:max_batch_len])
    return torch.tensor(input_ids), torch.tensor(labels)

# Load and split — 95/5, held out BEFORE shuffling further, fixed seed for reproducibility
all_texts = [json.loads(l)["text"] for l in open(STAGE_B_PATH, encoding="utf-8") if l.strip()]
random.Random(42).shuffle(all_texts)

val_size = int(0.05 * len(all_texts))
val_texts = all_texts[:val_size]
train_texts = all_texts[val_size:]

print(f"Train: {len(train_texts)}, Val: {len(val_texts)}")

train_ds = JsonlTextDataset(train_texts, tokenizer, CONFIG["max_seq_len"])
val_ds = JsonlTextDataset(val_texts, tokenizer, CONFIG["max_seq_len"])

from functools import partial
collate = partial(collate_fn, pad_id=tokenizer.eos_token_id, max_len=CONFIG["max_seq_len"])

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, collate_fn=collate)

Train: 77753, Val: 4092


In [11]:
# Sanity check: confirm labels are correctly shifted
batch = next(iter(train_loader))
input_ids, labels = batch
print("input_ids[0][:10]:", input_ids[0][:10].tolist())
print("labels[0][:10]:   ", labels[0][:10].tolist())
print("\nExpected: labels[0][:9] should equal input_ids[0][1:10]")
print("Match:", input_ids[0][1:10].tolist() == labels[0][:9].tolist())

input_ids[0][:10]: [21017, 30532, 25, 198, 12982, 25, 314, 765, 284, 1011]
labels[0][:10]:    [30532, 25, 198, 12982, 25, 314, 765, 284, 1011, 10022]

Expected: labels[0][:9] should equal input_ids[0][1:10]
Match: True


In [12]:
# Cell 6 — Optimizer, scheduler

In [13]:
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=CONFIG["lr"], betas=(0.9, 0.95))

total_steps = (len(train_loader) // CONFIG["grad_accum_steps"]) * CONFIG["epochs"]

def lr_lambda(step):
    if step < CONFIG["warmup_steps"]:
        return step / max(1, CONFIG["warmup_steps"])
    progress = (step - CONFIG["warmup_steps"]) / max(1, total_steps - CONFIG["warmup_steps"])
    return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.cuda.amp.GradScaler()

print(f"Total optimizer steps: {total_steps}")

Total optimizer steps: 12150


C:\Users\MIND & MATTER\AppData\Local\Temp\ipykernel_12128\347633844.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [14]:
# Cell 7 — Eval loop

In [15]:
@torch.no_grad()
def evaluate():
    model.eval()
    total_loss, total_batches = 0.0, 0
    for input_ids, labels in val_loader:
        input_ids, labels = input_ids.to(CONFIG["device"]), labels.to(CONFIG["device"])
        logits = model(input_ids)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100)
        total_loss += loss.item()
        total_batches += 1
    model.train()
    avg_loss = total_loss / max(1, total_batches)
    ppl = math.exp(avg_loss)
    return avg_loss, ppl

In [16]:
# Cell 8 — Checkpoint helpers

In [17]:
CKPT_DIR = os.path.join(BASE_DIR, "checkpoints", "stage_b")
os.makedirs(os.path.join(CKPT_DIR, "best"), exist_ok=True)

def save_checkpoint(path, step, epoch, best_val_loss):
    torch.save({
        "step": step,
        "epoch": epoch,
        "best_val_loss": best_val_loss,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
    }, path)

In [22]:
import os
for f in ["latest.pt", "best/best.pt"]:
    p = os.path.join(CKPT_DIR, f)
    if os.path.exists(p):
        os.remove(p)
        print(f"Removed: {p}")
    else:
        print(f"Not found (already clean): {p}")

Removed: C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\checkpoints\stage_b\latest.pt
Removed: C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\checkpoints\stage_b\best/best.pt


In [19]:
LATEST_CKPT = os.path.join(CKPT_DIR, "latest.pt")

def load_checkpoint_if_exists():
    if os.path.exists(LATEST_CKPT):
        print(f"Found existing checkpoint at {LATEST_CKPT}, resuming...")
        ckpt = torch.load(LATEST_CKPT, map_location=CONFIG["device"], weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        step = ckpt["step"]
        epoch = ckpt["epoch"]
        best_val_loss = ckpt["best_val_loss"]
        print(f"Resumed at step {step}, epoch {epoch}, best_val_loss {best_val_loss:.4f}")
        return step, epoch, best_val_loss
    else:
        print("No existing checkpoint found, starting fresh.")
        return 0, 0, float("inf")

start_step, start_epoch, best_val_loss = load_checkpoint_if_exists()
global_step = start_step
print(f"Will resume training from global_step={global_step}, epoch={start_epoch}")

No existing checkpoint found, starting fresh.
Will resume training from global_step=0, epoch=0


In [20]:
# Cell 9 — Training loop with tightened early stopping

In [21]:
model.train()
patience_counter = 0
PATIENCE = CONFIG["patience"]
LOG_EVERY = 50
GENERATE_EVERY = 500

SAMPLE_PROMPT = "### Instruction:\nWhat should I do this weekend?\n\n### Response:\n"

@torch.no_grad()
def generate_sample(prompt, max_new_tokens=60, repetition_penalty=1.3, window=128):
    model.eval()
    ids = tokenizer.encode(prompt)
    input_ids = torch.tensor([ids]).to(CONFIG["device"])
    generated = list(ids)

    for _ in range(max_new_tokens):
        input_ids_trunc = input_ids[:, -CONFIG["max_seq_len"]:]
        logits = model(input_ids_trunc)[:, -1, :]

        recent = generated[-window:]
        for tok_id in set(recent):
            logits[0, tok_id] /= repetition_penalty

        next_id = torch.argmax(logits, dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_id], dim=1)
        generated.append(next_id.item())
        if next_id.item() == tokenizer.eos_token_id:
            break

    model.train()
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

for epoch in range(start_epoch, CONFIG["epochs"]):
    epoch_start_time = time.time()

    for i, (input_ids, labels) in enumerate(train_loader):
        input_ids, labels = input_ids.to(CONFIG["device"]), labels.to(CONFIG["device"])

        logits = model(input_ids)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100)
        loss = loss / CONFIG["grad_accum_steps"]
        loss.backward()

        if (i + 1) % CONFIG["grad_accum_steps"] == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, CONFIG["grad_clip"])
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            step_loss = loss.item() * CONFIG["grad_accum_steps"]
            step_ppl = math.exp(min(step_loss, 20))
            current_lr = scheduler.get_last_lr()[0]

            if global_step % LOG_EVERY == 0:
                print(f"Epoch {epoch} | Step {global_step} | loss {step_loss:.4f} | ppl {step_ppl:.2f} | lr {current_lr:.2e}")

            if global_step % GENERATE_EVERY == 0:
                sample_output = generate_sample(SAMPLE_PROMPT)
                print(f"\n  [SAMPLE @ step {global_step}] {sample_output}\n")

            if global_step % CONFIG["eval_every_steps"] == 0:
                val_loss, val_ppl = evaluate()
                print(f"  [EVAL] Step {global_step} | val_loss {val_loss:.4f} | val_ppl {val_ppl:.2f}")

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    patience_counter = 0
                    save_checkpoint(os.path.join(CKPT_DIR, "best", "best.pt"), global_step, epoch, best_val_loss)
                    print("  -> new best, saved")
                else:
                    patience_counter += 1
                    print(f"  -> no improvement ({patience_counter}/{PATIENCE})")

                if patience_counter >= PATIENCE:
                    print("Early stopping triggered.")
                    save_checkpoint(LATEST_CKPT, global_step, epoch, best_val_loss)
                    break

            if global_step % CONFIG["save_every_steps"] == 0:
                save_checkpoint(LATEST_CKPT, global_step, epoch, best_val_loss)
    else:
        epoch_time = time.time() - epoch_start_time
        print(f"\n=== Epoch {epoch} complete in {epoch_time/60:.1f} min ===")
        sample_output = generate_sample(SAMPLE_PROMPT)
        print(f"Sample generation for prompt: {SAMPLE_PROMPT!r}")
        print(sample_output)
        print("=" * 60)
        save_checkpoint(LATEST_CKPT, global_step, epoch + 1, best_val_loss)
        continue
    break

print("Training complete.")

Epoch 0 | Step 50 | loss 5.8163 | ppl 335.71 | lr 5.00e-05
Epoch 0 | Step 100 | loss 4.8396 | ppl 126.42 | lr 1.00e-04
Epoch 0 | Step 150 | loss 5.7645 | ppl 318.79 | lr 1.50e-04
Epoch 0 | Step 200 | loss 5.1705 | ppl 176.00 | lr 2.00e-04
  [EVAL] Step 200 | val_loss 5.4362 | val_ppl 229.57
  -> new best, saved
Epoch 0 | Step 250 | loss 5.3662 | ppl 214.05 | lr 2.00e-04
Epoch 0 | Step 300 | loss 5.3877 | ppl 218.71 | lr 2.00e-04
Epoch 0 | Step 350 | loss 5.6995 | ppl 298.70 | lr 2.00e-04
Epoch 0 | Step 400 | loss 5.4807 | ppl 240.02 | lr 2.00e-04
  [EVAL] Step 400 | val_loss 5.3593 | val_ppl 212.57
  -> new best, saved
Epoch 0 | Step 450 | loss 5.0904 | ppl 162.46 | lr 2.00e-04
Epoch 0 | Step 500 | loss 5.7077 | ppl 301.19 | lr 2.00e-04

  [SAMPLE @ step 500] ### Instruction:
What should I do this weekend?

### Response:
I am going to go to the conference. It is a great day for me, and it will be my first week of work. We have been working on our new project since we were in the office

In [ ]:
@torch.no_grad()
def generate_sample_sampled(prompt, max_new_tokens=60, temperature=0.8, top_p=0.9):
    model.eval()
    ids = tokenizer.encode(prompt)
    input_ids = torch.tensor([ids]).to(CONFIG["device"])
    for _ in range(max_new_tokens):
        logits = model(input_ids[:, -CONFIG["max_seq_len"]:])[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cum_probs = torch.cumsum(sorted_probs, dim=-1)
        cutoff = (cum_probs > top_p).float().argmax(dim=-1).item()
        keep_idx = sorted_idx[0, :cutoff+1]
        keep_probs = sorted_probs[0, :cutoff+1]
        next_id = keep_idx[torch.multinomial(keep_probs, 1)].unsqueeze(0)
        input_ids = torch.cat([input_ids, next_id], dim=1)
        if next_id.item() == tokenizer.eos_token_id:
            break
    model.train()
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

print(generate_sample_sampled(SAMPLE_PROMPT))